In [1]:
import os
import random
import shutil
import hashlib
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

In [2]:

import tensorflow as tf
print(tf.__version__)

2.20.0


# image preprocessing

In [3]:
import os
import cv2
import numpy as np
import pandas as pd

# Path to train & test directories
train_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil_photos-20251020T144319Z-1-001\soil_photos\train"
test_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil_photos-20251020T144319Z-1-001\soil_photos\test"

# Define allowed soil classes
soil_classes = ['Alluvial soil', 'Black Soil', 'Red soil', 'Gravel', 'Clay soil', 'Silt']

# Function to check image validity
def check_images(folder_path):
    stats = {}
    for soil in soil_classes:
        soil_dir = os.path.join(folder_path, soil)
        if not os.path.exists(soil_dir):
            print(f"⚠️ Missing folder: {soil}")
            continue

        valid_count = 0
        for img_name in os.listdir(soil_dir):
            img_path = os.path.join(soil_dir, img_name)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    print(f"❌ Corrupt image: {img_path}")
                    continue
                valid_count += 1
            except Exception as e:
                print(f"❌ Error reading {img_name}: {e}")

        stats[soil] = valid_count
    return stats

print("🧹 Checking training images...")
train_stats = check_images(train_path)
print("\n📊 Training Image Counts:", train_stats)

print("\n🧹 Checking test images...")
test_stats = check_images(test_path)
print("\n📊 Test Image Counts:", test_stats)


🧹 Checking training images...

📊 Training Image Counts: {'Alluvial soil': 528, 'Black Soil': 231, 'Red soil': 264, 'Gravel': 25, 'Clay soil': 199, 'Silt': 25}

🧹 Checking test images...

📊 Test Image Counts: {'Alluvial soil': 54, 'Black Soil': 116, 'Red soil': 106, 'Gravel': 5, 'Clay soil': 65, 'Silt': 5}


# Resize & Normalize All Images

To feed into CNNs, images must have a fixed size (e.g., 150×150 or 224×224) and pixel values normalized to [0–1].

In [4]:
# import cv2
# import numpy as np
# import os
# from tqdm import tqdm

# # ===============================
# # 🗂️ Input & Output Folder Paths
# # ===============================
# train_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil_photos-20251020T144319Z-1-001\soil_photos\train"
# test_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil_photos-20251020T144319Z-1-001\soil_photos\test"

# clean_train = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Train"
# clean_test = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Test"
# os.makedirs(clean_train, exist_ok=True)
# os.makedirs(clean_test, exist_ok=True)

# IMG_SIZE = (150, 150)

# # ===============================
# # 🧹 Resize & Normalize Function
# # ===============================
# def resize_and_clean(src_folder, dest_folder):
#     """
#     Auto-detect all subfolders (soil classes), resize images to 150x150,
#     and normalize pixel values (0–1)
#     """
#     if not os.path.exists(src_folder):
#         print(f"❌ Source folder not found: {src_folder}")
#         return

#     soil_classes = [d for d in os.listdir(src_folder) if os.path.isdir(os.path.join(src_folder, d))]

#     if not soil_classes:
#         print(f"⚠️ No soil folders found inside: {src_folder}")
#         return

#     for soil in soil_classes:
#         src_path = os.path.join(src_folder, soil)
#         dest_path = os.path.join(dest_folder, soil)
#         os.makedirs(dest_path, exist_ok=True)

#         for img_name in tqdm(os.listdir(src_path), desc=f"Processing {soil}"):
#             try:
#                 if not img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
#                     continue  # Skip unsupported formats (e.g., .gif)

#                 img_path = os.path.join(src_path, img_name)
#                 img = cv2.imread(img_path)

#                 if img is None:
#                     print(f"⚠️ Skipped unreadable: {img_name}")
#                     continue

#                 # Resize
#                 img = cv2.resize(img, IMG_SIZE)


#                 cv2.imwrite(os.path.join(dest_path, img_name), img)



#                 # # Normalize
#                 # img_normalized = img / 255.0
#                 # img_uint8 = (img_normalized * 255).astype(np.uint8)

#                 # # Save cleaned image
#                 # cv2.imwrite(os.path.join(dest_path, img_name), img_uint8)

#             except Exception as e:
#                 print(f"⚠️ Skipped {img_name}: {e}")

# # ===============================
# # 🚀 Run Cleaning
# # ===============================
# print("\n🧼 Cleaning and normalizing training data...")
# resize_and_clean(train_path, clean_train)

# print("\n🧼 Cleaning and normalizing testing data...")
# resize_and_clean(test_path, clean_test)

# print("\n✅ All images cleaned, resized, and normalized successfully!")



🧼 Cleaning and normalizing training data...


Processing Silt: 100%|██████████| 25/25 [00:00<00:00, 594.67it/s]



🧼 Cleaning and normalizing testing data...


Processing Silt: 100%|██████████| 5/5 [00:00<00:00, 290.25it/s]


✅ All images cleaned, resized, and normalized successfully!


In [4]:
import cv2
import os
from tqdm import tqdm

# ===============================
# 🗂️ Input & Output Folder Paths
# ===============================
train_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil_photos-20251020T144319Z-1-001\soil_photos\train"
test_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil_photos-20251020T144319Z-1-001\soil_photos\test"

clean_train = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Train"
clean_test = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Test"
os.makedirs(clean_train, exist_ok=True)
os.makedirs(clean_test, exist_ok=True)

IMG_SIZE = (150, 150)

# ===============================
# 🧹 Resize & Save Function
# ===============================
def resize_and_clean(src_folder, dest_folder):
    """
    Auto-detect all subfolders (soil classes),
    resize images to 150x150,
    and save them without normalization.
    """
    if not os.path.exists(src_folder):
        print(f"❌ Source folder not found: {src_folder}")
        return

    soil_classes = [d for d in os.listdir(src_folder) if os.path.isdir(os.path.join(src_folder, d))]
    if not soil_classes:
        print(f"⚠️ No soil folders found inside: {src_folder}")
        return

    for soil in soil_classes:
        src_path = os.path.join(src_folder, soil)
        dest_path = os.path.join(dest_folder, soil)
        os.makedirs(dest_path, exist_ok=True)

        for img_name in tqdm(os.listdir(src_path), desc=f"Processing {soil}"):
            try:
                if not img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                    continue  # Skip unsupported formats

                img_path = os.path.join(src_path, img_name)
                img = cv2.imread(img_path)

                if img is None:
                    print(f"⚠️ Skipped unreadable: {img_name}")
                    continue

                # Resize
                img_resized = cv2.resize(img, IMG_SIZE)

                # Save with correct path and filename
                cv2.imwrite(os.path.join(dest_path, img_name), img_resized)

            except Exception as e:
                print(f"⚠️ Skipped {img_name}: {e}")

# ===============================
# 🚀 Run Cleaning
# ===============================
print("\n🧼 Cleaning and resizing training data...")
resize_and_clean(train_path, clean_train)

print("\n🧼 Cleaning and resizing testing data...")
resize_and_clean(test_path, clean_test)

print("\n✅ All images cleaned and resized successfully!")



🧼 Cleaning and resizing training data...


Processing Silt: 100%|██████████| 25/25 [00:00<00:00, 518.69it/s]



🧼 Cleaning and resizing testing data...


Processing Silt: 100%|██████████| 5/5 [00:00<00:00, 362.61it/s]


✅ All images cleaned and resized successfully!


# CSV Preprocessing

In [5]:
import pandas as pd

# Load dataset
csv_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil_photos-20251020T144319Z-1-001\soil_photos\Soil2_Vision_Weather_Season_Crop_Dataset.csv"
df = pd.read_csv(csv_path)

# Show all columns
print("📋 Columns in CSV:")
print(df.columns.tolist())

# Show first few rows
print("\n🔍 Data preview:")
print(df.head())


📋 Columns in CSV:
['Dataset', 'Soil_Type', 'Image_Path', 'Detected_Tone', 'pH Range', 'Temperature (°C)', 'Humidity (%)', 'Nitrogen (mg/kg)', 'Phosphorus (mg/kg)', 'Potassium (mg/kg)', 'Crop_Recommendations', 'Fertilizer_Recommendations', 'Weather_Based_Crop_Recommendation', 'Season_Based_Crop_Recommendation', 'Current_Season', 'Current_Temperature(°C)', 'Current_Humidity(%)', 'Weather_Description']

🔍 Data preview:
  Dataset      Soil_Type                                         Image_Path  \
0   train  Alluvial soil  C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PR...   
1   train  Alluvial soil  C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PR...   
2   train  Alluvial soil  C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PR...   
3   train  Alluvial soil  C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PR...   
4   train  Alluvial soil  C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PR...   

  Detected_Tone pH Range Temperature (°C) Humidity (%) Nitrogen (mg/kg)  \
0        Bluish  6.5–7.5      

In [6]:
import pandas as pd

# Load dataset
csv_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil_photos-20251020T144319Z-1-001\soil_photos\Soil2_Vision_Weather_Season_Crop_Dataset.csv"
df = pd.read_csv(csv_path)

# Clean text columns
df.columns = df.columns.str.strip()
df['Soil_Type'] = df['Soil_Type'].str.strip()

# Convert ranges to numeric min/max columns
def extract_range(val):
    if isinstance(val, str) and ("–" in val or "-" in val):
        val = val.replace("–", "-")
        parts = val.split("-")
        return float(parts[0]), float(parts[1])
    return None, None

# ✅ Updated column names (from your CSV)
for col in ['Nitrogen (mg/kg)', 'Phosphorus (mg/kg)', 'Potassium (mg/kg)', 'pH Range']:
    df[[f"{col}_min", f"{col}_max"]] = df[col].apply(lambda x: pd.Series(extract_range(x)))

print("✅ Processed Data Preview:")
print(df.head())


✅ Processed Data Preview:
  Dataset      Soil_Type                                         Image_Path  \
0   train  Alluvial soil  C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PR...   
1   train  Alluvial soil  C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PR...   
2   train  Alluvial soil  C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PR...   
3   train  Alluvial soil  C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PR...   
4   train  Alluvial soil  C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PR...   

  Detected_Tone pH Range Temperature (°C) Humidity (%) Nitrogen (mg/kg)  \
0        Bluish  6.5–7.5            25–35        60–80          280–450   
1       Reddish  6.5–7.5            25–35        60–80          280–450   
2       Reddish  6.5–7.5            25–35        60–80          280–450   
3       Reddish  6.5–7.5            25–35        60–80          280–450   
4       Reddish  6.5–7.5            25–35        60–80          280–450   

  Phosphorus (mg/kg) Potassium (mg/kg)  ... Curr

# Save Clean Dataset

In [7]:
clean_csv_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned3_Soil_Vision_Dataset.csv"
df.to_csv(clean_csv_path, index=False)
print(f"✅ Cleaned dataset saved at: {clean_csv_path}")


✅ Cleaned dataset saved at: C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned3_Soil_Vision_Dataset.csv


# Split and Load Preprocessed Data

In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Train"
test_dir  = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Test"

# Image generator with real-time augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical'
)


Found 1033 images belonging to 7 classes.
Found 256 images belonging to 7 classes.
Found 354 images belonging to 7 classes.


In [9]:
print("✅ Soil Classes:", train_generator.class_indices)


✅ Soil Classes: {'Alluvial soil': 0, 'Black Soil': 1, 'Clay soil': 2, 'Gravel': 3, 'Red soil': 4, 'Sand': 5, 'Silt': 6}


# Build the CNN model

In [9]:
# from tensorflow.keras import layers, models

# model = models.Sequential([
#     layers.Input(shape=(150,150,3)),
#     layers.Conv2D(32, (3,3), activation='relu'),
#     layers.MaxPooling2D(2,2),

#     layers.Conv2D(64, (3,3), activation='relu'),
#     layers.MaxPooling2D(2,2),

#     layers.Conv2D(128, (3,3), activation='relu'),
#     layers.MaxPooling2D(2,2),

#     layers.Flatten(),
#     layers.Dense(128, activation='relu'),
#     layers.Dropout(0.3),
#     layers.Dense(6, activation='softmax')   # ✅ Match 6 classes
# ])

# model.compile(
#     optimizer='adam',
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )


In [10]:
from tensorflow.keras import layers, models

# ✅ Check detected classes
num_classes = len(train_generator.class_indices)
print("✅ Detected Classes:", train_generator.class_indices)
print("Total Classes:", num_classes)

# ✅ Define CNN model
model = models.Sequential([
    layers.Input(shape=(150,150,3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    # ✅ Automatically match number of classes
    layers.Dense(num_classes, activation='softmax')
])

# ✅ Compile model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

✅ Detected Classes: {'Alluvial soil': 0, 'Black Soil': 1, 'Clay soil': 2, 'Gravel': 3, 'Red soil': 4, 'Sand': 5, 'Silt': 6}
Total Classes: 7


# Train the model

In [11]:
print("Training classes:", train_generator.class_indices)
print("Total classes:", len(train_generator.class_indices))


Training classes: {'Alluvial soil': 0, 'Black Soil': 1, 'Clay soil': 2, 'Gravel': 3, 'Red soil': 4, 'Sand': 5, 'Silt': 6}
Total classes: 7


In [12]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    verbose=1
)

Epoch 1/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 14s 342ms/step - accuracy: 0.5644 - loss: 1.3616 - val_accuracy: 0.6172 - val_loss: 1.1496
Epoch 2/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 313ms/step - accuracy: 0.7435 - loss: 0.8098 - val_accuracy: 0.7422 - val_loss: 0.7650
Epoch 3/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 306ms/step - accuracy: 0.7754 - loss: 0.6276 - val_accuracy: 0.6914 - val_loss: 0.8993
Epoch 4/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 312ms/step - accuracy: 0.7744 - loss: 0.6125 - val_accuracy: 0.7227 - val_loss: 0.7747
Epoch 5/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 316ms/step - accuracy: 0.7967 - loss: 0.5154 - val_accuracy: 0.7188 - val_loss: 0.6453
Epoch 6/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 319ms/step - accuracy: 0.8190 - loss: 0.4482 - val_accuracy: 0.6445 - val_loss: 0.7730
Epoch 7/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 311ms/step - accuracy: 0.8374 - loss: 0.4144 - val_accuracy: 0.7695 - val_loss: 0.5981
Epoch 8/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 11s 328ms/step - accuracy: 0.8412 - loss: 0.4236 - val_accu

# Test accuracy

In [13]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"✅ Test Accuracy: {test_acc*100:.2f}%")


12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.8842 - loss: 0.4559
✅ Test Accuracy: 88.42%


In [14]:
model.save(r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil3_vision_model_v1.h5")
print("✅ Model saved successfully!")


✅ Model saved successfully!


# Model Inference / Prediction

In [15]:
from tensorflow.keras.models import load_model
import numpy as np
from tensorflow.keras.preprocessing import image
import pandas as pd

# Load trained model
model = load_model(r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil3_vision_model_v1.h5")

# Define soil class labels (in same order as training folders)
class_labels = ['Alluvial soil', 'Black soil', 'Red soil', 'Clay soil', 'Silt soil', 'Gravel soil', 'Sand soil']


# MobileNetV2 Transfer Learning model

The version 2 of your soil vision project

Uses transfer learning (MobileNetV2)

Includes class weighting, augmentation, and fine-tuning

In [15]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import os

# ===============================================
# STEP 1️⃣ — Directory setup
# ===============================================
train_dir = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Train"
test_dir  = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Test"

# ===============================================
# STEP 2️⃣ — Image generators with strong augmentation
# ===============================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.3,
    shear_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    vertical_flip=True,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# ===============================================
# STEP 3️⃣ — Compute class weights for imbalance
# ===============================================
classes = list(train_gen.class_indices.keys())
y_train = train_gen.classes
class_weights = compute_class_weight(class_weight="balanced",
                                     classes=np.unique(y_train),
                                     y=y_train)
class_weights = dict(enumerate(class_weights))

print("\n⚖️ Computed Class Weights:")
for c, w in class_weights.items():
    print(f"{classes[c]}: {w:.2f}")

# ===============================================
# STEP 4️⃣ — Transfer Learning Model (MobileNetV2)
# ===============================================
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
base_model.trainable = False  # Freeze base layers initially

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(classes), activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Found 1033 images belonging to 7 classes.
Found 256 images belonging to 7 classes.
Found 354 images belonging to 7 classes.

⚖️ Computed Class Weights:
Alluvial soil: 0.35
Black Soil: 0.81
Clay soil: 0.92
Gravel: 7.38
Red soil: 0.70
Sand: 7.38
Silt: 7.38


C:\Users\Pranjal Yallurkar\AppData\Local\Temp\ipykernel_16272\263364072.py:74: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(150, 150, 3))


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [23]:
# # STEP 5️⃣ — Training
# # ===============================================
# callbacks = [
#     tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
#     tf.keras.callbacks.ModelCheckpoint(
#         r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soils_vision_model_v1.h5",
#         save_best_only=True
#     )
# ]

# history = model.fit(
#     train_gen,
#     validation_data=val_gen,
#     epochs=25,
#     class_weight=class_weights,
#     callbacks=callbacks
# )

# # ===============================================
# # STEP 6️⃣ — Fine-tune top MobileNet layers
# # ===============================================
# base_model.trainable = True
# for layer in base_model.layers[:-40]:  # Freeze most layers, train last 40
#     layer.trainable = False

# model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#               loss='categorical_crossentropy',
#               metrics=['accuracy'])

# fine_tune_history = model.fit(
#     train_gen,
#     validation_data=val_gen,
#     epochs=10,
#     class_weight=class_weights,
#     callbacks=callbacks
# )

# # ===============================================
# # STEP 7️⃣ — Evaluate on test data
# # ===============================================
# test_loss, test_acc = model.evaluate(test_gen)
# print(f"\n✅ Final Test Accuracy: {test_acc*100:.2f}%")

# # ===============================================
# # STEP 8️⃣ — Save final model
# # ===============================================
# model.save(r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soils_vision_model_v2_final.h5")
# print("\n💾 Model saved successfully!")

# # ===============================================
# # STEP 9️⃣ — Verify predicted classes
# # ===============================================
# print("\n🔍 Detected Classes:", classes)

C:\Users\Pranjal Yallurkar\AppData\Roaming\Python\Python312\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.2806 - loss: 2.5542

33/33 ━━━━━━━━━━━━━━━━━━━━ 21s 453ms/step - accuracy: 0.2801 - loss: 2.5456 - val_accuracy: 0.2617 - val_loss: 1.7805
Epoch 2/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step - accuracy: 0.3836 - loss: 1.4479

33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 392ms/step - accuracy: 0.3847 - loss: 1.4511 - val_accuracy: 0.3359 - val_loss: 1.6451
Epoch 3/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 301ms/step - accuracy: 0.4361 - loss: 1.2698

33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 388ms/step - accuracy: 0.4374 - loss: 1.2666 - val_accuracy: 0.5430 - val_loss: 1.3737
Epoch 4/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step - accuracy: 0.5106 - loss: 1.1792

...

33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 408ms/step - accuracy: 0.5470 - loss: 0.9315 - val_accuracy: 0.5469 - val_loss: 1.2993
Epoch 7/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step - accuracy: 0.6590 - loss: 0.7301

33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 386ms/step - accuracy: 0.6586 - loss: 0.7306 - val_accuracy: 0.5234 - val_loss: 1.2250
Epoch 8/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step - accuracy: 0.6243 - loss: 0.7786

33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 380ms/step - accuracy: 0.6250 - loss: 0.7778 - val_accuracy: 0.5938 - val_loss: 1.0718
Epoch 9/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 381ms/step - accuracy: 0.6453 - loss: 0.6903 - val_accuracy: 0.5742 - val_loss: 1.1415
Epoch 10/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 12s 383ms/step - accuracy: 0.6431 - loss: 0.7317 - val_accuracy: 0.5820 - val_loss: 1.0920
Epoch 11/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 380ms/step - accuracy: 0.6756 - loss: 0.7819 - val_accuracy: 0.5898 - val_loss: 1.1484
Epoch 12/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 382ms/step - accuracy: 0.6867 - loss: 0.6047 - val_accuracy: 0.5859 - val_loss: 1.2227
Epoch 13/25
33/33 ━━━━━━━━━━━━━━━━━━━━ 12s 374ms/step - accuracy: 0.7301 - loss: 0.5915 - val_accuracy: 0.5586 - val_loss: 1.2167
Epoch 1/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 448ms/step - accuracy: 0.3843 - loss: 2.3462

33/33 ━━━━━━━━━━━━━━━━━━━━ 38s 598ms/step - accuracy: 0.3852 - loss: 2.3467 - val_accuracy: 0.5977 - val_loss: 1.0623
Epoch 2/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 399ms/step - accuracy: 0.4425 - loss: 1.9350 - val_accuracy: 0.5117 - val_loss: 1.2064
Epoch 3/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 13s 406ms/step - accuracy: 0.4717 - loss: 1.4396 - val_accuracy: 0.5742 - val_loss: 1.0985
Epoch 4/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 14s 413ms/step - accuracy: 0.5302 - loss: 1.2734 - val_accuracy: 0.5352 - val_loss: 1.2351
Epoch 5/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 14s 408ms/step - accuracy: 0.5501 - loss: 1.3757 - val_accuracy: 0.4883 - val_loss: 1.2468
Epoch 6/10
33/33 ━━━━━━━━━━━━━━━━━━━━ 14s 422ms/step - accuracy: 0.5780 - loss: 1.2615 - val_accuracy: 0.4648 - val_loss: 1.3706
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 138ms/step - accuracy: 0.6608 - loss: 0.9358



✅ Final Test Accuracy: 70.34%

💾 Model saved successfully!

🔍 Detected Classes: ['Alluvial soil', 'Black Soil', 'Clay soil', 'Gravel', 'Red soil', 'Sand', 'Silt']


# Testing

In [16]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Train"

datagen = ImageDataGenerator(rescale=1./255)
train_gen = datagen.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='categorical'
)

print(train_gen.class_indices)


Found 1289 images belonging to 7 classes.
{'Alluvial soil': 0, 'Black Soil': 1, 'Clay soil': 2, 'Gravel': 3, 'Red soil': 4, 'Sand': 5, 'Silt': 6}


# Predict all testing

# Step 1 — Load Model and Predict Soil Type

In [17]:
import pandas as pd

# Load CSV file
csv_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned3_Soil_Vision_Dataset.csv"
df = pd.read_csv(csv_path)
print("🧾 Columns in CSV:\n", df.columns.tolist())



🧾 Columns in CSV:
 ['Dataset', 'Soil_Type', 'Image_Path', 'Detected_Tone', 'pH Range', 'Temperature (°C)', 'Humidity (%)', 'Nitrogen (mg/kg)', 'Phosphorus (mg/kg)', 'Potassium (mg/kg)', 'Crop_Recommendations', 'Fertilizer_Recommendations', 'Weather_Based_Crop_Recommendation', 'Season_Based_Crop_Recommendation', 'Current_Season', 'Current_Temperature(°C)', 'Current_Humidity(%)', 'Weather_Description', 'Nitrogen (mg/kg)_min', 'Nitrogen (mg/kg)_max', 'Phosphorus (mg/kg)_min', 'Phosphorus (mg/kg)_max', 'Potassium (mg/kg)_min', 'Potassium (mg/kg)_max', 'pH Range_min', 'pH Range_max']


# Alluvial soil

# Weather API

In [6]:
import requests

API_KEY = "d9c834bc3e00761992fc6cb1ab2e60bd"
CITY = "Belagavi"
url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"

response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    print(f"✅ Connected successfully! Current temp in {CITY}: {data['main']['temp']}°C")
else:
    print(f"⚠️ Failed to fetch weather data. Status: {response.status_code}")

✅ Connected successfully! Current temp in Belagavi: 22.07°C


In [7]:
import requests

API_KEY = "d9c834bc3e00761992fc6cb1ab2e60bd"  # paste your key here
CITY = "Belagavi"  # test with your city name

url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"

response = requests.get(url)
print("🔍 Status Code:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("\n🌤️ Weather:", data["weather"][0]["description"])
    print("🌡️ Temperature:", data["main"]["temp"], "°C")
    print("💧 Humidity:", data["main"]["humidity"], "%")
else:
    print("⚠️ Failed to fetch weather data. Status:", response.status_code)

🔍 Status Code: 200

🌤️ Weather: scattered clouds
🌡️ Temperature: 22.07 °C
💧 Humidity: 56 %


In [8]:
import requests
import time

API_KEY = "d9c834bc3e00761992fc6cb1ab2e60bd"
CITY = "Belagavi"

def get_weather():
    url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"
    data = requests.get(url).json()
    return data["main"]["temp"], data["main"]["humidity"]

temps, hums = [], []

# collect 3 samples every 10 seconds (example)
for _ in range(3):
    temp, hum = get_weather()
    temps.append(temp)
    hums.append(hum)
    time.sleep(10)

avg_temp = round(sum(temps)/len(temps), 2)
avg_hum = round(sum(hums)/len(hums), 2)

print(f"🌡️ Stable Temperature: {avg_temp} °C")
print(f"💧 Stable Humidity: {avg_hum} %")


🌡️ Stable Temperature: 22.07 °C
💧 Stable Humidity: 56.0 %


## TESTING

## 1 Alluvial soil

In [4]:
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import pandas as pd
import requests

# ==========================================================
# 🧠 STEP 1 — Load Model
# ==========================================================
model = load_model(
    r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soil3_vision_model_v1.h5"
)

# IMPORTANT: CLASS ORDER MUST MATCH MODEL TRAINING
class_labels = ['Alluvial soil', 'Black Soil', 'Clay soil', 'Gravel', 'Red soil', 'Silt', 'Sand']

# ==========================================================
# 🖼️ STEP 2 — Preprocess Input Image
# ==========================================================
img_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Test\Alluvial soil\Alluvial_4.jpg"

img = image.load_img(img_path, target_size=(150, 150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

# Predict
prediction = model.predict(img_array)
pred_index = np.argmax(prediction)
predicted_soil = class_labels[pred_index]

print("\n🧠 Predicted Soil Type:", predicted_soil)

# ==========================================================
# 🌿 STEP 3 — Load Soil Dataset
# ==========================================================
df = pd.read_csv(
    r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned3_Soil_Vision_Dataset.csv"
)

# Clean column names
df.columns = (
    df.columns.str.strip()
              .str.replace(" ", "_")
              .str.replace("-", "_")
              .str.replace("(", "")
              .str.replace(")", "")
)

df["Soil_Type"] = df["Soil_Type"].str.strip()

# ==========================================================
# 🧹 STEP 4 — Correct Soil Name Mapping
# ==========================================================
mapping = {
    "Gravel": "Gravel soil",
    "Silt": "Silty soil",
    "Sand": "Sandy soil",
    "Black Soil": "Black Soil", 
    "Red soil": "Red soil",
    "Clay soil": "Clay soil",
    "Alluvial soil": "Alluvial soil"
}

mapped_soil = mapping.get(predicted_soil, predicted_soil)
print("Mapped Name:", mapped_soil)

# ==========================================================
# 🔍 STEP 5 — Find Soil Info
# ==========================================================
soil_info = df[df['Soil_Type'].str.lower() == mapped_soil.lower()]

if soil_info.empty:
    print(f"⚠ No matching soil data found for: {mapped_soil}")
    exit()

row = soil_info.iloc[0]

# ==========================================================
# 📌 Print Soil Data
# ==========================================================
print("\n🌿 Soil Details:")
print(f"Soil Type: {row['Soil_Type']}")
print(f"pH Range: {row['pH_Range']}")
print(f"Nitrogen: {row['Nitrogen_mg/kg']}")
print(f"Phosphorus: {row['Phosphorus_mg/kg']}")
print(f"Potassium: {row['Potassium_mg/kg']}")

print("\n🌾 Recommended Crops:", row['Crop_Recommendations'])
print("\n💧 Recommended Fertilizers:", row['Fertilizer_Recommendations'])

# ==========================================================
# 🌦️ STEP 6 — Weather API
# ==========================================================
API_KEY = "d9c834bc3e00761992fc6cb1ab2e60bd"
CITY = "Belagavi"

url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"
res = requests.get(url)

if res.status_code == 200:
    data = res.json()
    print("\n🌦️ Weather:")
    print("Temperature:", data["main"]["temp"])
    print("Humidity:", data["main"]["humidity"])
    print("Sky:", data["weather"][0]["description"])
else:
    print("⚠ Weather API error")

# ==========================================================
# 🌾 STEP 7 — Weather-Based Crop Recommendation
# ==========================================================
weather_col = [c for c in df.columns if "weather" in c.lower()]

if weather_col:
    print("\n🌾 Weather-Based Crop Recommendation:")
    print(row[weather_col[0]])
else:
    print("⚠ No weather-based crop column found")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step

🧠 Predicted Soil Type: Sand
Mapped Name: Sandy soil

🌿 Soil Details:
Soil Type: Sandy soil
pH Range: 5.5–6.8
Nitrogen: 100–180
Phosphorus: 15–30
Potassium: 80–150

🌾 Recommended Crops: Coconut, Watermelon, Groundnut, Cashew, Carrot

💧 Recommended Fertilizers: Organic Manure, Compost, NPK, Urea, Biofertilizer

🌦️ Weather:
Temperature: 21.07
Humidity: 68
Sky: scattered clouds

🌾 Weather-Based Crop Recommendation:
Coconut, Watermelon, Groundnut, Cashew, Carrot


In [19]:
import tensorflow as tf
tf.get_logger().setLevel('ERROR')


# ==========================================================
# 🧠 STEP 1 — Predict Soil Type Using Trained CNN
# ==========================================================
from tensorflow.keras.models import load_model
import numpy as np
from tensorflow.keras.preprocessing import image
import pandas as pd
import requests

# Load your trained model
model = load_model(r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soils_vision_model_v1.h5")

# Input test image
img_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Test\Alluvial soil\Alluvial_4.jpg"

# Preprocess
img = image.load_img(img_path, target_size=(150,150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

# Class labels
class_labels = ['Alluvial soil', 'Black Soil', 'Clay soil', 'Gravel', 'Red soil', 'Silt', 'Sand']

# Predict soil type
prediction = model.predict(img_array)
predicted_soil = class_labels[np.argmax(prediction)]
print(f"\n🧠 Predicted Soil Type: {predicted_soil}")

# ==========================================================
# 🌿 STEP 2 — Fetch Soil Info from Dataset
# ==========================================================
dataset_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned1_Soil_Vision_Dataset.csv"
df = pd.read_csv(dataset_path)

# Clean column names for consistent access
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_")

df['Soil_Type'] = df['Soil_Type'].str.strip()

# Label correction mapping
label_mapping = {
    "Gravel": "Gravel soil",
    "Silt": "Silty soil",
    "Sand": "Sandy soil"
}

mapped_soil = label_mapping.get(predicted_soil, predicted_soil)

soil_info = df[df['Soil_Type'].str.lower() == mapped_soil.lower()]

if not soil_info.empty:
    row = soil_info.iloc[0]

    print("\n🌿 Soil Details:")
    print(f"Soil_Type: {row['Soil_Type']}")
    print(f"pH Range: {row['pH_Range']}")
    print(f"Temperature: {row['Temperature_(°C)']}")
    print(f"Humidity: {row['Humidity_(%)']}")
    print(f"Nitrogen: {row['Nitrogen_(mg/kg)']}")
    print(f"Phosphorus: {row['Phosphorus_(mg/kg)']}")
    print(f"Potassium: {row['Potassium_(mg/kg)']}")

    print("\n🌾 Recommended Crops:")
    print(row['Crop_Recommendations'])

    print("\n💧 Recommended Fertilizers:")
    print(row['Fertilizer_Recommendations'])
else:
    print(f"⚠️ No matching soil data found for: {mapped_soil}")

# ==========================================================
# 🌦️ STEP 3 — Fetch Live Weather Data (OpenWeatherMap)
# ==========================================================
API_KEY = "d9c834bc3e00761992fc6cb1ab2e60bd"
CITY = "Belagavi"

url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    weather_desc = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    humidity = data["main"]["humidity"]

    print(f"\n🌤️ Current Weather in {CITY}:")
    print(f"Temperature (°C): {temp}")
    print(f"Humidity (%): {humidity}")
    print(f"Weather: {weather_desc}")

    # ==========================================================
    # 🌾 STEP 4 — Weather-based Crop Recommendation (auto-detect)
    # ==========================================================
    possible_cols = [c for c in df.columns if "weather" in c.lower() and "recommend" in c.lower()]

    if possible_cols:
        col_name = possible_cols[0]
        print(f"\n🌦️ Weather-Based Crop Recommendation:")
        print(row[col_name])
    else:
        print("\n⚠️ Weather-Based Crop Recommendation column not found in dataset.")
else:
    print("⚠️ Failed to fetch weather data.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step

🧠 Predicted Soil Type: Alluvial soil

🌿 Soil Details:
Soil_Type: Alluvial soil
pH Range: 6.5–7.5
Temperature: 25–35
Humidity: 60–80
Nitrogen: 280–450
Phosphorus: 40–60
Potassium: 300–450

🌾 Recommended Crops:
Rice, Wheat, Maize, Sugarcane, Pulses

💧 Recommended Fertilizers:
DAP, NPK, Ammonium Sulphate, Compost, Vermicompost

🌤️ Current Weather in Belagavi:
Temperature (°C): 26.07
Humidity (%): 53
Weather: haze

🌦️ Weather-Based Crop Recommendation:
Rice, Wheat


## 2 Black soil

In [3]:
import tensorflow as tf
tf.get_logger().setLevel('ERROR')


# ==========================================================
# 🧠 STEP 1 — Predict Soil Type Using Trained CNN
# ==========================================================
from tensorflow.keras.models import load_model
import numpy as np
from tensorflow.keras.preprocessing import image
import pandas as pd
import requests

# Load your trained model
model = load_model(r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\backend\model\soil2_vision_model_v1.h5")

# Input test image
img_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Test\Black Soil\Black_2.jpg"

# Preprocess
img = image.load_img(img_path, target_size=(150,150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

# Class labels
class_labels = ['Alluvial soil', 'Black Soil', 'Clay soil', 'Gravel', 'Red soil', 'Silt', 'Sand']

# Predict soil type
prediction = model.predict(img_array)
predicted_soil = class_labels[np.argmax(prediction)]
print(f"\n🧠 Predicted Soil Type: {predicted_soil}")

# ==========================================================
# 🌿 STEP 2 — Fetch Soil Info from Dataset
# ==========================================================
dataset_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\backend\model\Cleaned2_Soil_Vision_Dataset.csv"
df = pd.read_csv(dataset_path)

# Clean column names for consistent access
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_")

df['Soil_Type'] = df['Soil_Type'].str.strip()

# Label correction mapping
label_mapping = {
    "Gravel": "Gravel soil",
    "Silt": "Silty soil",
    "Sand": "Sandy soil"
}

mapped_soil = label_mapping.get(predicted_soil, predicted_soil)

soil_info = df[df['Soil_Type'].str.lower() == mapped_soil.lower()]

if not soil_info.empty:
    row = soil_info.iloc[0]

    print("\n🌿 Soil Details:")
    print(f"Soil_Type: {row['Soil_Type']}")
    print(f"pH Range: {row['pH_Range']}")
    print(f"Temperature: {row['Temperature_(°C)']}")
    print(f"Humidity: {row['Humidity_(%)']}")
    print(f"Nitrogen: {row['Nitrogen_(mg/kg)']}")
    print(f"Phosphorus: {row['Phosphorus_(mg/kg)']}")
    print(f"Potassium: {row['Potassium_(mg/kg)']}")

    print("\n🌾 Recommended Crops:")
    print(row['Crop_Recommendations'])

    print("\n💧 Recommended Fertilizers:")
    print(row['Fertilizer_Recommendations'])
else:
    print(f"⚠️ No matching soil data found for: {mapped_soil}")

# ==========================================================
# 🌦️ STEP 3 — Fetch Live Weather Data (OpenWeatherMap)
# ==========================================================
API_KEY = "d9c834bc3e00761992fc6cb1ab2e60bd"
CITY = "Belagavi"

url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    weather_desc = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    humidity = data["main"]["humidity"]

    print(f"\n🌤️ Current Weather in {CITY}:")
    print(f"Temperature (°C): {temp}")
    print(f"Humidity (%): {humidity}")
    print(f"Weather: {weather_desc}")

    # ==========================================================
    # 🌾 STEP 4 — Weather-based Crop Recommendation (auto-detect)
    # ==========================================================
    possible_cols = [c for c in df.columns if "weather" in c.lower() and "recommend" in c.lower()]

    if possible_cols:
        col_name = possible_cols[0]
        print(f"\n🌦️ Weather-Based Crop Recommendation:")
        print(row[col_name])
    else:
        print("\n⚠️ Weather-Based Crop Recommendation column not found in dataset.")
else:
    print("⚠️ Failed to fetch weather data.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step

🧠 Predicted Soil Type: Clay soil

🌿 Soil Details:
Soil_Type: Clay soil
pH Range: 6.0–7.0
Temperature: 20–30
Humidity: 65–80
Nitrogen: 220–350
Phosphorus: 25–40
Potassium: 250–400

🌾 Recommended Crops:
Rice, Sugarcane, Lettuce, Broccoli, Peas

💧 Recommended Fertilizers:
Compost, NPK, Urea, Biofertilizer, Vermicompost

🌤️ Current Weather in Belagavi:
Temperature (°C): 22.07
Humidity (%): 64
Weather: scattered clouds

🌦️ Weather-Based Crop Recommendation:
Rice, Sugarcane, Lettuce, Broccoli, Peas


## 3 Clay soil

In [1]:
import tensorflow as tf
tf.get_logger().setLevel('ERROR')


# ==========================================================
# 🧠 STEP 1 — Predict Soil Type Using Trained CNN
# ==========================================================
from tensorflow.keras.models import load_model
import numpy as np
from tensorflow.keras.preprocessing import image
import pandas as pd
import requests

# Load your trained model
model = load_model(r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\backend\model\soil2_vision_model_v1.h5")

# Input test image
img_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Test\Clay soil\Clay_5.jpg"

# Preprocess
img = image.load_img(img_path, target_size=(150,150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

# Class labels
class_labels = ['Alluvial soil', 'Black Soil', 'Clay soil', 'Gravel', 'Red soil', 'Silt', 'Sand']

# Predict soil type
prediction = model.predict(img_array)
predicted_soil = class_labels[np.argmax(prediction)]
print(f"\n🧠 Predicted Soil Type: {predicted_soil}")

# ==========================================================
# 🌿 STEP 2 — Fetch Soil Info from Dataset
# ==========================================================
dataset_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\backend\model\Cleaned2_Soil_Vision_Dataset.csv"
df = pd.read_csv(dataset_path)

# Clean column names for consistent access
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_")

df['Soil_Type'] = df['Soil_Type'].str.strip()

# Label correction mapping
label_mapping = {
    "Gravel": "Gravel soil",
    "Silt": "Silty soil",
    "Sand": "Sandy soil"
}

mapped_soil = label_mapping.get(predicted_soil, predicted_soil)

soil_info = df[df['Soil_Type'].str.lower() == mapped_soil.lower()]

if not soil_info.empty:
    row = soil_info.iloc[0]

    print("\n🌿 Soil Details:")
    print(f"Soil_Type: {row['Soil_Type']}")
    print(f"pH Range: {row['pH_Range']}")
    print(f"Temperature: {row['Temperature_(°C)']}")
    print(f"Humidity: {row['Humidity_(%)']}")
    print(f"Nitrogen: {row['Nitrogen_(mg/kg)']}")
    print(f"Phosphorus: {row['Phosphorus_(mg/kg)']}")
    print(f"Potassium: {row['Potassium_(mg/kg)']}")

    print("\n🌾 Recommended Crops:")
    print(row['Crop_Recommendations'])

    print("\n💧 Recommended Fertilizers:")
    print(row['Fertilizer_Recommendations'])
else:
    print(f"⚠️ No matching soil data found for: {mapped_soil}")

# ==========================================================
# 🌦️ STEP 3 — Fetch Live Weather Data (OpenWeatherMap)
# ==========================================================
API_KEY = "d9c834bc3e00761992fc6cb1ab2e60bd"
CITY = "Belagavi"

url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    weather_desc = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    humidity = data["main"]["humidity"]

    print(f"\n🌤️ Current Weather in {CITY}:")
    print(f"Temperature (°C): {temp}")
    print(f"Humidity (%): {humidity}")
    print(f"Weather: {weather_desc}")

    # ==========================================================
    # 🌾 STEP 4 — Weather-based Crop Recommendation (auto-detect)
    # ==========================================================
    possible_cols = [c for c in df.columns if "weather" in c.lower() and "recommend" in c.lower()]

    if possible_cols:
        col_name = possible_cols[0]
        print(f"\n🌦️ Weather-Based Crop Recommendation:")
        print(row[col_name])
    else:
        print("\n⚠️ Weather-Based Crop Recommendation column not found in dataset.")
else:
    print("⚠️ Failed to fetch weather data.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step

🧠 Predicted Soil Type: Clay soil

🌿 Soil Details:
Soil_Type: Clay soil
pH Range: 6.0–7.0
Temperature: 20–30
Humidity: 65–80
Nitrogen: 220–350
Phosphorus: 25–40
Potassium: 250–400

🌾 Recommended Crops:
Rice, Sugarcane, Lettuce, Broccoli, Peas

💧 Recommended Fertilizers:
Compost, NPK, Urea, Biofertilizer, Vermicompost

🌤️ Current Weather in Belagavi:
Temperature (°C): 22.07
Humidity (%): 64
Weather: scattered clouds

🌦️ Weather-Based Crop Recommendation:
Rice, Sugarcane, Lettuce, Broccoli, Peas


## 4 Gravel

In [22]:
import tensorflow as tf
tf.get_logger().setLevel('ERROR')


# ==========================================================
# 🧠 STEP 1 — Predict Soil Type Using Trained CNN
# ==========================================================
from tensorflow.keras.models import load_model
import numpy as np
from tensorflow.keras.preprocessing import image
import pandas as pd
import requests

# Load your trained model
model = load_model(r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soils_vision_model_v1.h5")

# Input test image
img_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Test\Gravel\2.jpg"

# Preprocess
img = image.load_img(img_path, target_size=(150,150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

# Class labels
class_labels = ['Alluvial soil', 'Black Soil', 'Clay soil', 'Gravel', 'Red soil', 'Silt', 'Sand']

# Predict soil type
prediction = model.predict(img_array)
predicted_soil = class_labels[np.argmax(prediction)]
print(f"\n🧠 Predicted Soil Type: {predicted_soil}")

# ==========================================================
# 🌿 STEP 2 — Fetch Soil Info from Dataset
# ==========================================================
dataset_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned1_Soil_Vision_Dataset.csv"
df = pd.read_csv(dataset_path)

# Clean column names for consistent access
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_")

df['Soil_Type'] = df['Soil_Type'].str.strip()

# Label correction mapping
label_mapping = {
    "Gravel": "Gravel soil",
    "Silt": "Silty soil",
    "Sand": "Sandy soil"
}

mapped_soil = label_mapping.get(predicted_soil, predicted_soil)

soil_info = df[df['Soil_Type'].str.lower() == mapped_soil.lower()]

if not soil_info.empty:
    row = soil_info.iloc[0]

    print("\n🌿 Soil Details:")
    print(f"Soil_Type: {row['Soil_Type']}")
    print(f"pH Range: {row['pH_Range']}")
    print(f"Temperature: {row['Temperature_(°C)']}")
    print(f"Humidity: {row['Humidity_(%)']}")
    print(f"Nitrogen: {row['Nitrogen_(mg/kg)']}")
    print(f"Phosphorus: {row['Phosphorus_(mg/kg)']}")
    print(f"Potassium: {row['Potassium_(mg/kg)']}")

    print("\n🌾 Recommended Crops:")
    print(row['Crop_Recommendations'])

    print("\n💧 Recommended Fertilizers:")
    print(row['Fertilizer_Recommendations'])
else:
    print(f"⚠️ No matching soil data found for: {mapped_soil}")

# ==========================================================
# 🌦️ STEP 3 — Fetch Live Weather Data (OpenWeatherMap)
# ==========================================================
API_KEY = "d9c834bc3e00761992fc6cb1ab2e60bd"
CITY = "Belagavi"

url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    weather_desc = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    humidity = data["main"]["humidity"]

    print(f"\n🌤️ Current Weather in {CITY}:")
    print(f"Temperature (°C): {temp}")
    print(f"Humidity (%): {humidity}")
    print(f"Weather: {weather_desc}")

    # ==========================================================
    # 🌾 STEP 4 — Weather-based Crop Recommendation (auto-detect)
    # ==========================================================
    possible_cols = [c for c in df.columns if "weather" in c.lower() and "recommend" in c.lower()]

    if possible_cols:
        col_name = possible_cols[0]
        print(f"\n🌦️ Weather-Based Crop Recommendation:")
        print(row[col_name])
    else:
        print("\n⚠️ Weather-Based Crop Recommendation column not found in dataset.")
else:
    print("⚠️ Failed to fetch weather data.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

🧠 Predicted Soil Type: Gravel

🌿 Soil Details:
Soil_Type: Gravel soil
pH Range: 6.0–7.5
Temperature: 25–35
Humidity: 40–60
Nitrogen: 120–200
Phosphorus: 10–20
Potassium: 80–150

🌾 Recommended Crops:
Millet, Cotton, Pulses, Castor, Jowar

💧 Recommended Fertilizers:
Potash, NPK, Compost, Organic Manure, Biofertilizer

🌤️ Current Weather in Belagavi:
Temperature (°C): 28.07
Humidity (%): 47
Weather: few clouds

🌦️ Weather-Based Crop Recommendation:
Millet


## 5 Red soil

In [23]:
import tensorflow as tf
tf.get_logger().setLevel('ERROR')


# ==========================================================
# 🧠 STEP 1 — Predict Soil Type Using Trained CNN
# ==========================================================
from tensorflow.keras.models import load_model
import numpy as np
from tensorflow.keras.preprocessing import image
import pandas as pd
import requests

# Load your trained model
model = load_model(r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\soils_vision_model_v1.h5")

# Input test image
img_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned_Test\Red soil\Copy of images122.jpg"

# Preprocess
img = image.load_img(img_path, target_size=(150,150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

# Class labels
class_labels = ['Alluvial soil', 'Black Soil', 'Clay soil', 'Gravel', 'Red soil', 'Silt', 'Sand']

# Predict soil type
prediction = model.predict(img_array)
predicted_soil = class_labels[np.argmax(prediction)]
print(f"\n🧠 Predicted Soil Type: {predicted_soil}")

# ==========================================================
# 🌿 STEP 2 — Fetch Soil Info from Dataset
# ==========================================================
dataset_path = r"C:\Users\Pranjal Yallurkar\OneDrive\Desktop\PROJECT\Final\Cleaned1_Soil_Vision_Dataset.csv"
df = pd.read_csv(dataset_path)

# Clean column names for consistent access
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("-", "_")

df['Soil_Type'] = df['Soil_Type'].str.strip()

# Label correction mapping
label_mapping = {
    "Gravel": "Gravel soil",
    "Silt": "Silty soil",
    "Sand": "Sandy soil"
}

mapped_soil = label_mapping.get(predicted_soil, predicted_soil)

soil_info = df[df['Soil_Type'].str.lower() == mapped_soil.lower()]

if not soil_info.empty:
    row = soil_info.iloc[0]

    print("\n🌿 Soil Details:")
    print(f"Soil_Type: {row['Soil_Type']}")
    print(f"pH Range: {row['pH_Range']}")
    print(f"Temperature: {row['Temperature_(°C)']}")
    print(f"Humidity: {row['Humidity_(%)']}")
    print(f"Nitrogen: {row['Nitrogen_(mg/kg)']}")
    print(f"Phosphorus: {row['Phosphorus_(mg/kg)']}")
    print(f"Potassium: {row['Potassium_(mg/kg)']}")

    print("\n🌾 Recommended Crops:")
    print(row['Crop_Recommendations'])

    print("\n💧 Recommended Fertilizers:")
    print(row['Fertilizer_Recommendations'])
else:
    print(f"⚠️ No matching soil data found for: {mapped_soil}")

# ==========================================================
# 🌦️ STEP 3 — Fetch Live Weather Data (OpenWeatherMap)
# ==========================================================
API_KEY = "d9c834bc3e00761992fc6cb1ab2e60bd"
CITY = "Belagavi"

url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    weather_desc = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    humidity = data["main"]["humidity"]

    print(f"\n🌤️ Current Weather in {CITY}:")
    print(f"Temperature (°C): {temp}")
    print(f"Humidity (%): {humidity}")
    print(f"Weather: {weather_desc}")

    # ==========================================================
    # 🌾 STEP 4 — Weather-based Crop Recommendation (auto-detect)
    # ==========================================================
    possible_cols = [c for c in df.columns if "weather" in c.lower() and "recommend" in c.lower()]

    if possible_cols:
        col_name = possible_cols[0]
        print(f"\n🌦️ Weather-Based Crop Recommendation:")
        print(row[col_name])
    else:
        print("\n⚠️ Weather-Based Crop Recommendation column not found in dataset.")
else:
    print("⚠️ Failed to fetch weather data.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

🧠 Predicted Soil Type: Red soil

🌿 Soil Details:
Soil_Type: Red soil
pH Range: 6.0–7.0
Temperature: 24–34
Humidity: 45–65
Nitrogen: 180–300
Phosphorus: 20–40
Potassium: 200–350

🌾 Recommended Crops:
Groundnut, Millet, Maize, Pulses, Ragi

💧 Recommended Fertilizers:
DAP, NPK, Compost, Organic Manure, Biofertilizer

🌤️ Current Weather in Belagavi:
Temperature (°C): 28.07
Humidity (%): 47
Weather: few clouds

🌦️ Weather-Based Crop Recommendation:
Groundnut
